# DemandWise — Baselines e Modelos Supervisionados

Este notebook reúne as Etapas 4 e 5 do projeto: validação temporal, comparação de baselines e avaliação de modelos supervisionados. Os últimos 90 dias do treino simulam o horizonte futuro do `test.csv`.

## Estratégia sem vazamento temporal

- **Treino:** todas as datas anteriores aos últimos 90 dias.
- **Validação:** os últimos 90 dias, equivalentes ao horizonte do teste.
- **Origem única:** todos os baselines são calculados no último dia do treino.
- **Sem atualização:** as vendas reais da validação nunca atualizam as previsões.

Essa configuração mede uma previsão genuína de 90 passos à frente.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

project_candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(
    (path for path in project_candidates if (path / "src" / "train_model.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluate_model import evaluate_by_group, evaluate_models
from src.train_model import (
    BASELINE_COLUMNS,
    create_baseline_predictions,
    load_training_data,
    temporal_split,
    validate_baseline_predictions,
)

pio.templates.default = "plotly_white"
COLORS = {"primary": "#2563EB", "secondary": "#0F766E", "accent": "#F59E0B", "grid": "#E5E7EB"}

def style_figure(fig, title, height=470, show_legend=False):
    fig.update_layout(
        title={"text": title, "x": 0.01, "xanchor": "left"},
        height=height, showlegend=show_legend, separators=",.",
        margin={"l": 70, "r": 35, "t": 80, "b": 65},
        font={"family": "Arial, sans-serif", "size": 13},
        paper_bgcolor="white", plot_bgcolor="white",
    )
    fig.update_xaxes(showgrid=False, automargin=True)
    fig.update_yaxes(gridcolor=COLORS["grid"], zeroline=False, automargin=True)
    return fig

print(f"Raiz do projeto: {PROJECT_ROOT.resolve()}")

## 1. Divisão temporal

In [ ]:
data = load_training_data()
train, validation, validation_start = temporal_split(data, validation_days=90)

split_summary = pd.DataFrame(
    {
        "Período": ["Treino", "Validação"],
        "Data inicial": [train["date"].min(), validation["date"].min()],
        "Data final": [train["date"].max(), validation["date"].max()],
        "Dias": [train["date"].nunique(), validation["date"].nunique()],
        "Observações": [len(train), len(validation)],
        "Séries loja-produto": [
            train[["store", "item"]].drop_duplicates().shape[0],
            validation[["store", "item"]].drop_duplicates().shape[0],
        ],
    }
)
display(split_summary)

## 2. Geração dos baselines

Os modelos comparados são:

1. média global de todas as vendas do treino;
2. média histórica específica de cada loja-produto;
3. média dos últimos 7 dias de cada loja-produto;
4. média dos últimos 28 dias de cada loja-produto.

In [ ]:
predictions = create_baseline_predictions(train, validation)
validate_baseline_predictions(train, validation, predictions)

preview_columns = ["date", "store", "item", "sales", *BASELINE_COLUMNS.values()]
display(predictions[preview_columns].head())
print(f"Previsões geradas: {len(predictions):,}")

## 3. Comparação por métricas

- **MAE:** erro absoluto médio em unidades.
- **RMSE:** penaliza mais fortemente erros grandes.
- **MAPE:** erro percentual médio; vendas reais iguais a zero são ignoradas.
- **SMAPE:** erro percentual simétrico, mais estável quando os volumes são baixos.

In [ ]:
metrics = evaluate_models(predictions, BASELINE_COLUMNS)
display(metrics.style.format({"MAE": "{:.3f}", "RMSE": "{:.3f}", "MAPE": "{:.2f}%", "SMAPE": "{:.2f}%"}))

In [ ]:
absolute_metrics = metrics.melt(
    id_vars=["model"], value_vars=["MAE", "RMSE"],
    var_name="Métrica", value_name="Erro",
)
fig = px.bar(
    absolute_metrics, x="model", y="Erro", color="Métrica", barmode="group",
    text_auto=".2f", color_discrete_sequence=[COLORS["primary"], COLORS["secondary"]],
    labels={"model": "Baseline", "Erro": "Erro em unidades vendidas"},
)
style_figure(fig, "Comparação dos erros absolutos", height=500, show_legend=True)
fig.update_layout(legend={"orientation": "h", "y": 1.08, "x": 0})
fig.update_xaxes(tickangle=-15)
fig.update_yaxes(rangemode="tozero")
fig.update_traces(hovertemplate="%{x}<br>%{fullData.name}: %{y:.3f}<extra></extra>")
fig.show()

In [ ]:
percentage_metrics = metrics.melt(
    id_vars=["model"], value_vars=["MAPE", "SMAPE"],
    var_name="Métrica", value_name="Erro percentual",
)
fig = px.bar(
    percentage_metrics, x="model", y="Erro percentual", color="Métrica", barmode="group",
    text_auto=".1f", color_discrete_sequence=[COLORS["accent"], COLORS["primary"]],
    labels={"model": "Baseline", "Erro percentual": "Erro (%)"},
)
style_figure(fig, "Comparação dos erros percentuais", height=500, show_legend=True)
fig.update_layout(legend={"orientation": "h", "y": 1.08, "x": 0})
fig.update_xaxes(tickangle=-15)
fig.update_yaxes(rangemode="tozero", ticksuffix="%")
fig.update_traces(hovertemplate="%{x}<br>%{fullData.name}: %{y:.2f}%<extra></extra>")
fig.show()

## 4. Real versus previsto pelo melhor baseline

A visão diária agregada evidencia o que um baseline fixo consegue explicar e quais padrões temporais ainda permanecem sem modelagem.

In [ ]:
best_model = metrics.iloc[0]["model"]
best_prediction_column = BASELINE_COLUMNS[best_model]
daily_comparison = predictions.groupby("date", as_index=False)[["sales", best_prediction_column]].sum()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=daily_comparison["date"], y=daily_comparison["sales"],
    mode="lines", name="Real", line={"color": COLORS["primary"], "width": 2.5},
    hovertemplate="%{x|%d/%m/%Y}<br>Real: %{y:,.0f}<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=daily_comparison["date"], y=daily_comparison[best_prediction_column],
    mode="lines", name="Previsto", line={"color": COLORS["accent"], "width": 2.5, "dash": "dash"},
    hovertemplate="%{x|%d/%m/%Y}<br>Previsto: %{y:,.0f}<extra></extra>",
))
style_figure(fig, f"Real versus previsto — {best_model}", height=510, show_legend=True)
fig.update_layout(hovermode="x unified", legend={"orientation": "h", "y": 1.08, "x": 0})
fig.update_xaxes(title="Data")
fig.update_yaxes(title="Unidades vendidas por dia", rangemode="tozero")
fig.show()

## 5. Erro do melhor baseline por loja e produto

In [ ]:
store_metrics = evaluate_by_group(predictions, best_prediction_column, "store", best_model)
item_metrics = evaluate_by_group(predictions, best_prediction_column, "item", best_model)

fig = px.bar(
    store_metrics.sort_values("MAE", ascending=False), x="store", y="MAE",
    text_auto=".2f", color_discrete_sequence=[COLORS["secondary"]],
    labels={"store": "Loja", "MAE": "MAE (unidades)"},
)
style_figure(fig, f"MAE por loja — {best_model}", height=460)
fig.update_xaxes(type="category")
fig.update_yaxes(rangemode="tozero")
fig.show()

display(item_metrics.sort_values("MAE", ascending=False).head(10).round(3))

## 6. Conclusões

In [ ]:
best = metrics.iloc[0]
second = metrics.iloc[1]
relative_gain = (second["MAE"] / best["MAE"] - 1) * 100
worst_store = store_metrics.loc[store_metrics["MAE"].idxmax()]
worst_item = item_metrics.loc[item_metrics["MAE"].idxmax()]

conclusions = [
    f"O melhor baseline foi '{best['model']}', com MAE de {best['MAE']:.3f} e SMAPE de {best['SMAPE']:.2f}%.",
    f"Seu MAE foi {relative_gain:.1f}% menor que o segundo colocado.",
    "A segmentação por loja e produto é essencial: a média global perde diferenças estruturais entre as séries.",
    "As médias recentes de 7 e 28 dias não capturam adequadamente todo o horizonte de 90 dias quando permanecem fixas na origem.",
    f"A loja com maior MAE foi a Loja {int(worst_store['store'])}, com {worst_store['MAE']:.3f} unidades.",
    f"O produto com maior MAE foi o Produto {int(worst_item['item'])}, com {worst_item['MAE']:.3f} unidades.",
    "O futuro modelo supervisionado deverá superar esse baseline no mesmo corte temporal para demonstrar ganho real.",
]

for index, conclusion in enumerate(conclusions, start=1):
    print(f"{index}. {conclusion}")

## 7. Modelos supervisionados

Os modelos `RandomForestRegressor`, `GradientBoostingRegressor` e `HistGradientBoostingRegressor` foram treinados com as mesmas 19 features e avaliados no mesmo horizonte. A previsão é recursiva: cada novo dia usa apenas o histórico anterior e previsões já produzidas.

Execute `python src/train_ml_models.py` antes desta seção para gerar os artefatos.

In [ ]:
ML_PREDICTIONS_PATH = PROJECT_ROOT / "data" / "processed" / "ml_validation_predictions.csv"
MODEL_METRICS_PATH = PROJECT_ROOT / "reports" / "model_metrics.csv"
TRAINING_TIMES_PATH = PROJECT_ROOT / "reports" / "ml_training_times.csv"
FEATURE_IMPORTANCE_PATH = PROJECT_ROOT / "reports" / "ml_feature_importance.csv"

for artifact_path in [ML_PREDICTIONS_PATH, MODEL_METRICS_PATH, TRAINING_TIMES_PATH, FEATURE_IMPORTANCE_PATH]:
    if not artifact_path.exists():
        raise FileNotFoundError(f"Artefato ausente: {artifact_path}. Execute 'python src/train_ml_models.py'.")

ml_predictions = pd.read_csv(ML_PREDICTIONS_PATH, parse_dates=["date"])
model_ranking = pd.read_csv(MODEL_METRICS_PATH)
training_times = pd.read_csv(TRAINING_TIMES_PATH)
feature_importance = pd.read_csv(FEATURE_IMPORTANCE_PATH)
display(model_ranking.style.format({"MAE": "{:.3f}", "RMSE": "{:.3f}", "MAPE": "{:.2f}%", "SMAPE": "{:.2f}%"}))
display(training_times[["model", "training_rows", "training_seconds"]].style.format({"training_rows": "{:,.0f}", "training_seconds": "{:.1f} s"}))

In [ ]:
ranking_plot = model_ranking.sort_values("MAE", ascending=True)
type_colors = {"Supervisionado": COLORS["primary"], "Baseline": "#94A3B8"}
fig = px.bar(
    ranking_plot, x="MAE", y="model", color="model_type", orientation="h",
    text_auto=".2f", color_discrete_map=type_colors,
    labels={"model": "Modelo", "model_type": "Tipo", "MAE": "MAE (unidades)"},
)
style_figure(fig, "Ranking geral — modelos supervisionados versus baselines", height=540, show_legend=True)
fig.update_layout(legend={"orientation": "h", "y": 1.08, "x": 0})
fig.update_xaxes(rangemode="tozero")
fig.show()

## 8. Resultado recursivo do melhor modelo

In [ ]:
best_supervised = model_ranking[model_ranking["model_type"] == "Supervisionado"].iloc[0]
supervised_columns = {
    "Random Forest": "prediction_random_forest",
    "Gradient Boosting": "prediction_gradient_boosting",
    "HistGradient Boosting": "prediction_hist_gradient_boosting",
}
best_ml_column = supervised_columns[best_supervised["model"]]
daily_ml = ml_predictions.groupby("date", as_index=False)[["sales", best_ml_column]].sum()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=daily_ml["date"], y=daily_ml["sales"], mode="lines", name="Real",
    line={"color": COLORS["primary"], "width": 2.5},
    hovertemplate="%{x|%d/%m/%Y}<br>Real: %{y:,.0f}<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=daily_ml["date"], y=daily_ml[best_ml_column], mode="lines", name="Previsto",
    line={"color": COLORS["accent"], "width": 2.5, "dash": "dash"},
    hovertemplate="%{x|%d/%m/%Y}<br>Previsto: %{y:,.0f}<extra></extra>",
))
style_figure(fig, f"Real versus previsto — {best_supervised['model']}", height=510, show_legend=True)
fig.update_layout(hovermode="x unified", legend={"orientation": "h", "y": 1.08, "x": 0})
fig.update_xaxes(title="Data")
fig.update_yaxes(title="Unidades vendidas por dia", rangemode="tozero")
fig.show()

## 9. Importância das features

A importância nativa indica quais variáveis mais contribuíram para as divisões das árvores. Ela descreve o modelo, mas não deve ser interpretada isoladamente como causalidade.

In [ ]:
best_importance = (
    feature_importance[feature_importance["model"] == best_supervised["model"]]
    .nlargest(12, "importance")
    .sort_values("importance")
)
fig = px.bar(
    best_importance, x="importance", y="feature", orientation="h",
    text_auto=".3f", color_discrete_sequence=[COLORS["secondary"]],
    labels={"feature": "Feature", "importance": "Importância relativa"},
)
style_figure(fig, f"Principais features — {best_supervised['model']}", height=520)
fig.update_xaxes(rangemode="tozero")
fig.show()

## 10. Conclusões da modelagem supervisionada

In [ ]:
best_baseline_row = model_ranking[model_ranking["model_type"] == "Baseline"].iloc[0]
mae_gain = (best_baseline_row["MAE"] - best_supervised["MAE"]) / best_baseline_row["MAE"] * 100
supervised_conclusions = [
    f"{best_supervised['model']} foi o melhor modelo, com MAE de {best_supervised['MAE']:.3f} e SMAPE de {best_supervised['SMAPE']:.2f}%.",
    f"O ganho de MAE sobre o melhor baseline foi de {mae_gain:.1f}%.",
    "Os três modelos supervisionados superaram todos os baselines no horizonte recursivo.",
    "O HistGradient Boosting treinou mais rapidamente, mas perdeu precisão para os outros dois modelos.",
    "A avaliação recursiva fornece uma estimativa mais realista do uso no teste do que features calculadas com vendas reais futuras.",
]
for index, conclusion in enumerate(supervised_conclusions, start=1):
    print(f"{index}. {conclusion}")

## 11. Previsões futuras

O Random Forest foi retreinado até 31/12/2017 e aplicado recursivamente aos 90 dias do `test.csv`. Execute `python src/predict.py` antes desta seção.

In [ ]:
FINAL_FORECAST_PATH = PROJECT_ROOT / "data" / "processed" / "test_predictions.csv"
SUBMISSION_PATH = PROJECT_ROOT / "submissions" / "demandwise_submission.csv"
if not FINAL_FORECAST_PATH.exists() or not SUBMISSION_PATH.exists():
    raise FileNotFoundError("Execute 'python src/predict.py' para gerar as previsões finais.")

future_forecast = pd.read_csv(FINAL_FORECAST_PATH, parse_dates=["date"])
submission = pd.read_csv(SUBMISSION_PATH)
future_summary = pd.DataFrame(
    {
        "Indicador": ["Período", "Previsões", "Demanda total", "Média diária", "Valores ausentes"],
        "Valor": [
            f"{future_forecast['date'].min():%d/%m/%Y} a {future_forecast['date'].max():%d/%m/%Y}",
            f"{len(future_forecast):,}",
            f"{future_forecast['sales'].sum():,.0f}",
            f"{future_forecast.groupby('date')['sales'].sum().mean():,.0f}",
            int(submission['sales'].isna().sum()),
        ],
    }
)
display(future_summary)
display(submission.head())

In [ ]:
daily_future = future_forecast.groupby("date", as_index=False)["sales"].sum()
fig = px.line(
    daily_future, x="date", y="sales", markers=True,
    color_discrete_sequence=[COLORS["primary"]],
    labels={"date": "Data", "sales": "Unidades previstas"},
)
style_figure(fig, "Demanda diária prevista — janeiro a março de 2018", height=500)
fig.update_traces(
    line={"width": 2.5},
    hovertemplate="%{x|%d/%m/%Y}<br>Previsão: %{y:,.0f}<extra></extra>",
)
fig.update_yaxes(rangemode="tozero")
fig.show()

In [ ]:
monthly_future = (
    future_forecast.assign(month=future_forecast["date"].dt.to_period("M").astype(str))
    .groupby("month", as_index=False)["sales"].sum()
)
fig = px.bar(
    monthly_future, x="month", y="sales", text_auto=",.0f",
    color_discrete_sequence=[COLORS["secondary"]],
    labels={"month": "Mês", "sales": "Unidades previstas"},
)
style_figure(fig, "Demanda prevista por mês", height=450)
fig.update_yaxes(rangemode="tozero")
fig.show()

## Próxima etapa

Consolidar histórico, desempenho dos modelos e previsões futuras em um dashboard e produzir o relatório executivo com recomendações de estoque e planejamento.